In [1]:
# for auto-reloading modules
%load_ext autoreload
%autoreload 2

In [2]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA

from project_utils import *
from project_utils.dataset_utils import *

# Open plt figs in separate interactive windows: https://stackoverflow.com/a/14277370
%matplotlib qt

# POD-based Control
**Notebook Tasks**
1. Collect $X_{train}$, find its dimensionality, *plot in 3D*
2. Find how many PC Modes correspond to 70%, 80%, 90%, and 95% energy with a *cumulative energy plot*
3. *Plot train samples in 2D and 3D PC space*
4. Calculate the covariance matrix and threshold the correlation matrix to get dynamics $A$ in latent space
5. Find the minimum number of nodes needed to control the system

## 1. Collect $X_{train}$, find its shape, *plot as 4D animations*

In [3]:
train_folder = './data/train/'
test_folder = './data/test/'

JUMPING, RUNNING, WALKING = 0, 1, 2

COLORS = {JUMPING:(1,0,0,1), RUNNING:(0,1,0,1), WALKING:(0,0,1,1)}

if REMOVE_OUTLIERS := False:
  train_fnames = [
    'jumping_1', 'jumping_2', 'jumping_3', 'jumping_4', 'jumping_5',
    'running_1', 'running_3', 'running_4', 'running_5',
    'walking_2', 'walking_3', 'walking_4', 'walking_5'
  ]
  train_labels = [
    JUMPING, JUMPING, JUMPING, JUMPING, JUMPING,
    RUNNING, RUNNING, RUNNING, RUNNING,
    WALKING, WALKING, WALKING, WALKING
  ]
else:
  train_fnames = [
    'jumping_1', 'jumping_2', 'jumping_3', 'jumping_4', 'jumping_5',
    'running_1', 'running_2', 'running_3', 'running_4', 'running_5',
    'walking_1', 'walking_2', 'walking_3', 'walking_4', 'walking_5'
  ]

  train_labels = [
    JUMPING, JUMPING, JUMPING, JUMPING, JUMPING,
    RUNNING, RUNNING, RUNNING, RUNNING, RUNNING,
    WALKING, WALKING, WALKING, WALKING, WALKING
  ]

num_jumping = sum(1 if l == JUMPING else 0 for l in train_labels)
num_running = sum(1 if l == RUNNING else 0 for l in train_labels)
num_walking = sum(1 if l == WALKING else 0 for l in train_labels)

run_idx = num_jumping
walk_idx = num_jumping+num_running

test_fnames = ['jumping_1t', 'running_1t', 'walking_1t']

test_labels = [JUMPING, RUNNING, WALKING]

In [4]:
train_data_mat, test_data_mat = get_train_test_set(train_folder, train_fnames, test_folder, test_fnames, True)
print('train data shape', train_data_mat.shape, '|', 'test data shape', test_data_mat.shape)

train data shape (15, 11400) | test data shape (3, 11400)


In [5]:
# plot train data as animations (cell takes 7.75 minutes to run when SAVE_ACT_ANIMS)
if SAVE_ACT_ANIMS := False:
  for i in range(train_data_mat.shape[0]):
    if i < run_idx:
      plot_action(train_data_mat[i, :], f'Jumping_{i+1}')
    elif i < walk_idx:
      plot_action(train_data_mat[i, :], f'Running_{i-run_idx+1}')
    else:
      plot_action(train_data_mat[i, :], f'Walking_{i-walk_idx+1}')

  for i in range(test_data_mat.shape[0]):
	  plot_action(test_data_mat[i, :], f'Test_{i+1}')

### 1.b Create animations for first 5 Principal Components of $X_{train}$

In [6]:
pca_5D = PCA(5)
pca_5D.fit(train_data_mat);

In [7]:
# plot first 5 POD modes as animations (cell takes 2.2 minutes to run when SAVE_5_POD_ANIMS)
if SAVE_5_POD_ANIMS := True:
  for i in range(5):
    plot_action(pca_5D.inverse_transform(pca_5D.transform(pca_5D.components_[i:i+1, :])), f'POD_Mode_{i+1}')

In [8]:
# plot average activity projections (cell takes 1 minute to run when SAVE_MEAN_ACT_ANIMS)
reconstructed_jumping = pca_5D.inverse_transform(pca_5D.transform(train_data_mat[:run_idx,:]).mean(axis=0, keepdims=True))
reconstructed_running = pca_5D.inverse_transform(pca_5D.transform(train_data_mat[run_idx:walk_idx,:]).mean(axis=0, keepdims=True))
reconstructed_walking = pca_5D.inverse_transform(pca_5D.transform(train_data_mat[walk_idx:,:].mean(axis=0, keepdims=True)))

if SAVE_MEAN_ACT_ANIMS := True:
  plot_action(reconstructed_jumping, 'Low_Dim_Jumping')
  plot_action(reconstructed_running, 'Low_Dim_Running')
  plot_action(reconstructed_walking, 'Low_Dim_Walking')

del pca_5D

## 2. Find how many PC Modes correspond to 70%, 80%, 90%, and 95% energy with a *cumulative energy plot*

In [9]:
pca = PCA()
pca.fit(train_data_mat)
print('Effective rank:', int(sum(pca.singular_values_ > 1e-6)))
print('Singular values:', [float(round(sig, 2)) for sig in pca.singular_values_])

Effective rank: 14
Singular values: [28912.44, 18458.96, 15151.68, 10598.02, 8925.15, 7712.64, 7550.71, 6107.41, 4969.75, 4635.93, 3866.2, 3344.16, 2303.22, 915.41, 0.0]


In [10]:
energy = pca.singular_values_**2/np.sum(pca.singular_values_**2)
cumsum_energy = np.cumsum(energy)

cumsum_energy

array([0.45667356, 0.6428182 , 0.76823564, 0.82959569, 0.87311351,
       0.90561047, 0.93675719, 0.95713465, 0.97062754, 0.98236869,
       0.99053459, 0.99664415, 0.9995422 , 1.        , 1.        ])

In [11]:
idx_70 = np.argmax(cumsum_energy >= 0.7)
print(f'70% Var explained by {idx_70+1} PC Modes')
idx_80 = np.argmax(cumsum_energy >= 0.8)
print(f'80% Var explained by {idx_80+1} PC Modes')
idx_90 = np.argmax(cumsum_energy >= 0.9)
print(f'90% Var explained by {idx_90+1} PC Modes')
idx_95 = np.argmax(cumsum_energy >= 0.95)
print(f'95% Var explained by {idx_95+1} PC Modes')

plt.plot(cumsum_energy, c='k')

plt.hlines(0.7, xmin=0, xmax=energy.shape[0]-1, colors='tab:blue', linestyles=':', label=f'70% Var explained by {idx_70+1} PC Modes')
plt.hlines(0.8, xmin=0, xmax=energy.shape[0]-1, colors='tab:orange', linestyles=':', label=f'80% Var explained by {idx_80+1} PC Modes')
plt.hlines(0.9, xmin=0, xmax=energy.shape[0]-1, colors='tab:green', linestyles=':', label=f'90% Var explained by {idx_90+1} PC Modes')
plt.hlines(0.95, xmin=0, xmax=energy.shape[0]-1, colors='tab:red', linestyles=':', label=f'95% Var explained by {idx_95+1} PC Modes')
plt.xlabel('PC Mode index $j$')
plt.ylabel('$E_j$')
plt.title(f'Energy vs Singular Value Index (method = POD)')
plt.legend()
plt.savefig('./figs/PCA_Energy.png')
plt.close();

70% Var explained by 3 PC Modes
80% Var explained by 4 PC Modes
90% Var explained by 6 PC Modes
95% Var explained by 8 PC Modes


## 3. *Plot train samples in 2D and 3D PC space*

In [12]:
fig = plt.figure()
ax = fig.add_subplot()

pca2 = PCA(2)
X_train_2Dproj = pca2.fit_transform(train_data_mat).T

# plot the data vector tails
ax.scatter(X_train_2Dproj[0, :], X_train_2Dproj[1, :], c=[COLORS[c] for c in train_labels])
# plot the data vector shafts
for i in range(X_train_2Dproj.shape[1]):
    ax.plot([0, X_train_2Dproj[0, i]], [0, X_train_2Dproj[1, i]], c=COLORS[train_labels[i]])
# plot the jumping, running, and walking centroids
ax.scatter([X_train_2Dproj[0, :run_idx].mean()], [X_train_2Dproj[1, :run_idx].mean()], color=COLORS[0], marker='x')
ax.scatter([X_train_2Dproj[0, run_idx:walk_idx].mean()], [X_train_2Dproj[1, run_idx:walk_idx].mean()], color=COLORS[1], marker='x')
ax.scatter([X_train_2Dproj[0, walk_idx:].mean()], [X_train_2Dproj[1, walk_idx:].mean()], color=COLORS[2], marker='x')

max_ax_val = np.max(np.abs(X_train_2Dproj))
ax.set_xlim(-max_ax_val, max_ax_val)
ax.set_xlabel('PC 1')
ax.set_ylim(-max_ax_val, max_ax_val)
ax.set_ylabel('PC 2')
# Custom legend: https://stackoverflow.com/a/39500357
run_p, jump_p, walk_p = mpatches.Patch(color=COLORS[JUMPING], label='Jumping'), mpatches.Patch(color=COLORS[RUNNING], label='Running'), mpatches.Patch(color=COLORS[WALKING], label='Walking')
plt.legend(handles=[run_p, jump_p, walk_p])
plt.title('Projection of Train Data onto Top 2 POD Modes')
plt.show()
plt.savefig('./figs/PCA_proj2.png')
plt.close()
del pca2

In [13]:
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

pca3 = PCA(3)
X_train_3Dproj = pca3.fit_transform(train_data_mat).T

# 3D matplotlib: https://matplotlib.org/stable/gallery/mplot3d/2dcollections3d.html#sphx-glr-gallery-mplot3d-2dcollections3d-py
# plot the data vector tails
ax.scatter(X_train_3Dproj[0, :], X_train_3Dproj[1, :], zs=X_train_3Dproj[2, :], c=[COLORS[c] for c in train_labels])
# plot the data vector shafts
for i in range(X_train_3Dproj.shape[1]):
    ax.plot([0, X_train_3Dproj[0, i]], [0, X_train_3Dproj[1, i]], zs=[0, X_train_3Dproj[2, i]], zdir='z', c=COLORS[train_labels[i]])
# plot the jumping, running, and walking centroids
ax.scatter([X_train_3Dproj[0, :run_idx].mean()], [X_train_3Dproj[1, :run_idx].mean()], [X_train_3Dproj[2, :run_idx].mean()], color=COLORS[0], marker='x')
ax.scatter([X_train_3Dproj[0, run_idx:walk_idx].mean()], [X_train_3Dproj[1, run_idx:walk_idx].mean()], [X_train_3Dproj[2, run_idx:walk_idx].mean()], color=COLORS[1], marker='x')
ax.scatter([X_train_3Dproj[0, walk_idx:].mean()], [X_train_3Dproj[1, walk_idx:].mean()], [X_train_3Dproj[2, walk_idx:].mean()], color=COLORS[2], marker='x')

max_ax_val = np.max(np.abs(X_train_3Dproj))
ax.set_xlim3d(-max_ax_val, max_ax_val)
ax.set_xlabel('PC 1')
ax.set_ylim3d(-max_ax_val, max_ax_val)
ax.set_ylabel('PC 2')
ax.set_zlim3d(-max_ax_val, max_ax_val)
ax.set_zlabel('PC 3')
# Custom legend: https://stackoverflow.com/a/39500357
run_p, jump_p, walk_p = mpatches.Patch(color=(1,0,0,1), label='Jumping'), mpatches.Patch(color=(0,1,0,1), label='Running'), mpatches.Patch(color=(0,0,1,1), label='Walking')
plt.legend(handles=[run_p, jump_p, walk_p])
plt.title('Projection of Train Data onto Top 3 PC Modes')
plt.show()
plt.savefig('./figs/PCA_proj3.png')
plt.close()

del pca3

## 4. Calculate the covariance matrix and threshold the correlation matrix to get dynamics $A$ in latent space

In [18]:
k = idx_95+1
pca_k = PCA(k)
train_kD_proj = pca_k.fit_transform(train_data_mat).T

Cov_mat = train_kD_proj@train_kD_proj.T
Corr_mat = (1/np.linalg.norm(Cov_mat, axis=0, keepdims=True))*Cov_mat*(1/np.linalg.norm(Cov_mat, axis=1, keepdims=True))

In [36]:
max_abs = np.max(np.abs(Cov_mat))
fig, ax = plt.subplots()
cax = ax.imshow(Corr_mat[::-1, ::-1], 'RdBu_r', origin='upper')
ax.set_title('Correlation Matrix of Latent Projections (method=POD)')
fig.colorbar(cax, shrink=0.6, label='Corr(Z_i, Z_j)')
fig.tight_layout()
fig.show()

fig.savefig('./figs/PCA_corr.png')
plt.close()

In [38]:
fig = plt.figure()
ax = fig.add_subplot()
cax = ax.matshow(train_kD_proj, interpolation='nearest')
fig.colorbar(cax, shrink=0.5)

labels = ['']*train_kD_proj.shape[1]
labels[2], labels[7], labels[12] = 'jumping', 'running', 'walking'
ax.tick_params(top=False, labeltop=True, bottom=True, labelbottom=False)
ax.set_xticks(np.arange(train_kD_proj.shape[1]), labels=labels)
ax.set_ylabel(f'Coordinate in {k}D POD Space')
plt.title(f'Train Samples Projected onto k={k} POD Modes\n')
plt.tight_layout()
plt.show()

plt.savefig('./figs/PCA_samples_proj.png')
plt.close()

In [39]:
jump_centroid = np.mean(train_kD_proj[:, :run_idx], axis=1)
run_centroid = np.mean(train_kD_proj[:, run_idx:walk_idx], axis=1)
walk_centroid = np.mean(train_kD_proj[:, walk_idx:], axis=1)

centroids = np.stack((jump_centroid, run_centroid, walk_centroid))

fig = plt.figure(figsize=(6,6))
ax = fig.add_subplot()
cax = ax.matshow(centroids.T, interpolation='nearest')
fig.colorbar(cax, shrink=0.8)

CLASS_NAMES = ['jumping', 'running', 'walking']
ax.set_xticks(np.arange(3), CLASS_NAMES)
ax.set_ylabel(f'Coordinate in {k}D POD Space')
plt.title(f'Train Class Centroids in {k}D POD Space\n')
plt.tight_layout()
plt.show()

plt.savefig('./figs/PCA_centroids_proj.png')
plt.close()

## 5. Find the minimum number of nodes needed to control the system

In [ ]:
# UwU